# Customer Orders — New vs Returning Customers Per Day

**Question:** Given a `customer_orders` table with `order_id`, `customer_id`, `order_date`, and `order_amount`, write a query (in both SQL and PySpark) that shows for each order date how many orders came from **new customers** (first-ever purchase) versus **returning customers** (had ordered before).

In [0]:
%sql
-- Step 1: Create the customer_orders table with order_id, customer_id, order_date, order_amount
create or replace table b_sql.b_practice.customer_orders (
order_id integer,
customer_id integer,
order_date date,
order_amount integer
);

insert into b_sql.b_practice.customer_orders values(1,100,cast('2022-01-01' as date),2000),(2,200,cast('2022-01-01' as date),2500),(3,300,cast('2022-01-01' as date),2100)
,(4,100,cast('2022-01-02' as date),2000),(5,400,cast('2022-01-02' as date),2200),(6,500,cast('2022-01-02' as date),2700)
,(7,100,cast('2022-01-03' as date),3000),(8,400,cast('2022-01-03' as date),1000),(9,600,cast('2022-01-03' as date),3000)
;
-- Step 2: Insert 9 sample orders across 3 days (2022-01-01 to 2022-01-03) for 6 customers
insert into b_sql.b_practice.customer_orders values(1,100,cast('2022-01-01' as date),2000),(2,200,cast('2022-01-01' as date),2500),(3,300,cast('2022-01-01' as date),2100)
,(4,100,cast('2022-01-02' as date),2000),(5,400,cast('2022-01-02' as date),2200),(6,500,cast('2022-01-02' as date),2700)
,(7,100,cast('2022-01-03' as date),3000),(8,400,cast('2022-01-03' as date),1000),(9,600,cast('2022-01-03' as date),3000)
;
-- Step 3: Verify the inserted data
select * from b_sql.b_practice.customer_orders;

In [0]:
%sql
-- SQL Solution: Count new vs returning customers per order date
--
-- The CTE uses a window function to find each customer's first order date (min_date).
-- A flag is set to 1 when the order_date equals that customer's min_date (i.e., first purchase = new customer),
-- and 0 otherwise (returning customer).
-- The outer query groups by order_date and computes:
--   new  = sum of flag (count of first-time customers that day)
--   old  = count(*) - sum(flag) (count of returning customers that day)
with cte as (select  *, min(order_date) over(partition by customer_id order by order_date) as min_date, case when order_date= min(order_date) over(partition by customer_id order by order_date) then 1 else 0 end as flag  from b_sql.b_practice.customer_orders order by customer_id)
select order_date,sum(flag) as new, count(*)-sum(flag) as old from cte group by order_date

In [0]:
# Import common PySpark SQL functions, types, and Window specification
# (Window is needed for partitionBy/orderBy window functions used in Cell 5)
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# Load the customer_orders table from Unity Catalog into a Spark DataFrame
# Guarded with tableExists check in case the table hasn't been created yet
if spark.catalog.tableExists("b_sql.b_practice.customer_orders"):
    df_order=spark.read.table("b_sql.b_practice.customer_orders")
    df_order.display()
else:
    print("table not exist")

In [0]:
# PySpark Solution: Same new-vs-returning logic as the SQL CTE, using DataFrame API
#
# Step 1: Define a window partitioned by customer_id, ordered by order_date
# Step 2: Add min_order_date column = each customer's earliest order date
# Step 3: Flag = 1 when order_date equals min_order_date (new customer), else 0 (returning)
# Step 4: Group by order_date and aggregate:
#   total_order = count(*)
#   new         = sum(flag)  — first-time customers that day
#   old         = total_order - new — returning customers that day
min_date=Window.partitionBy(col("customer_id")).orderBy(col("order_date"))
df_order=df_order.withColumn("min_order_date",min(col("order_date")).over(min_date)).orderBy("customer_id")
df_flag=df_order.withColumn("flag",when(col("order_date")==col("min_order_date"),1).otherwise(0))
df_final=df_flag.groupBy("order_date").agg(count("*").alias("total_order"),sum("flag").alias("new"),(count("*")-sum("flag")).alias("old"))
df_final.display()